# Pass 1 — Production Run (Message Batches API)

**How it works:** Submit all ~5,680 fund requests as a single batch.  
Anthropic processes them asynchronously (usually <1 hour, guaranteed <24 hours).  
Come back later and download results.

**Why this over async:**
- **50% cheaper** — Sonnet 4 batch pricing: $1.50/MTok input, $7.50/MTok output
- **No rate-limit management** — Anthropic handles concurrency internally
- **No babysitting** — submit, close laptop, retrieve when done
- **No lost work** — results stored server-side for 29 days

**Workflow:**  
1. Run Sections 1–5 to submit the batch (~2–5 min to upload)  
2. Wait (check status with Section 6 whenever you want)  
3. Run Sections 7–9 to download and save results

**Limit:** 100,000 requests or 256 MB per batch. Your ~5,680 funds fit easily.

In [2]:
# === Section 1: Imports & Config ===

import pandas as pd
import time, os, json, re
import anthropic
from pathlib import Path
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

with open("Claude_API.txt", "r") as file:
    api_key = file.read().strip()
os.environ['ANTHROPIC_API_KEY'] = api_key

config = {}
with open("File_Directory.txt", "r") as file:
    for line in file:
        if ":" in line:
            key, value = line.split(":", 1)
            config[key.strip()] = value.strip()

INPUT_FILE = Path(config["Input"])
OUTPUT_DIR = Path(config["Output"])

MODEL = "claude-sonnet-4-6" # UPDATE as needed

OBJECTIVE_COLUMNS = [
    'PRIIPS KID Objective',
    'KIID Objective/Investment Policy',
    'Prospectus Objective',
    'Investment Strategy - English',
    'PRIIPS KID Objective - Danish',
    'PRIIPS KID Objective - Dutch',
    'PRIIPS KID Objective - Finnish',
    'PRIIPS KID Objective - French',
    'PRIIPS KID Objective - German',
    'PRIIPS KID Objective - Italian',
    'PRIIPS KID Objective - Norwegian',
    'PRIIPS KID Objective - Portuguese',
    'PRIIPS KID Objective - Spanish',
    'PRIIPS KID Objective - Swedish',
    'KIID Objective/Investment Policy - German',
    'KIID Objective/Investment Policy - French',
    'KIID Objective/Investment Policy - Italian',
    'KIID Objective/Investment Policy - Spanish',
    'KIID Objective/Investment Policy - Norwegian',
    'KIID Objective/Investment Policy - Swedish',
    'KIID Objective/Investment Policy - Finnish',
    'KIID Objective/Investment Policy - Portuguese',
    'KIID Objective/Investment Policy - Danish',
    'Investment Strategy - Danish',
    'Investment Strategy - Finnish',
    'Investment Strategy - French',
    'Investment Strategy - German',
    'Investment Strategy - Italian',
    'Investment Strategy - Norwegian',
    'Investment Strategy - Portuguese',
    'Investment Strategy - Spanish',
    'Investment Strategy - Swedish',
    'Strategy Description'
]

client = anthropic.Anthropic()

In [3]:
PASS1_SYSTEM_PROMPT = """You are extracting fund objectives from regulatory disclosure text for European mutual funds.

You will receive MULTIPLE columns of text for a single fund. Extract objectives from EACH column INDEPENDENTLY.
Do NOT cross-reference between columns. Treat each column as a standalone source.

WHAT IS A FUND OBJECTIVE:
The fund objective is the statement of what the fund aims to achieve for its investors — its goal or intended outcome.
Examples: long-term capital growth, regular income, maximizing total returns, beating a benchmark, matching an index.

MULTIPLE OBJECTIVES:
There may be more than one objective per column. Extract ALL and label them separately.
Split objectives when joined by "and", "while", "which also", "that also", or similar.
If one part states a financial goal and another states a sustainability goal, split them.
Examples:
- "provide income and moderate capital growth" → two objectives
- "exceed the performance of the index while maintaining a higher ESG score" → two objectives
- "seek long-term capital growth while reducing the risk of capital loss" → two objectives

Sustainability objectives (extract as separate objectives):
- Reducing greenhouse gas emissions, increasing biodiversity, improving living standards, advancing UN SDGs
- "while maintaining a higher ESG score than the index" or "lower carbon intensity" = separate objective
- Maintaining a minimum share of sustainable investments
- Specific solidarity or social investment commitments (e.g. "invest 5-10% in solidarity enterprises")
- Integration of good governance and sustainable development criteria as a fund-level goal

DO NOT INCLUDE:
- Investment policy/strategy: what the fund invests in, how securities are selected, asset allocation
- Mechanism through which objective is achieved ("by investing in...", "through active management", "through a quality asset strategy")
- Types of companies invested in ("invest in companies that...", "companies whose products...")
- Even if the sentence says "sustainable investment objective", extract only the fund's own intended outcome, not company activities
- Key distinction: A fund-level commitment ("invest 5-10% of assets in solidarity enterprises") IS an objective. A company description ("invest in companies that contribute to the SDGs") is NOT.
- "while taking into account ESG criteria" or "taking into account the risk level" = not an objective
- Risk information, distribution/dividend policy
- Benchmark references used solely for comparison (not as a target to beat)
- Duplicate objectives within the same column

SUSTAINABILITY CONTENT — WHAT TO EXTRACT VS WHAT TO EXCLUDE:

This is the most important judgment call in the extraction. Apply these rules in order:

Step 1 — Is it pure SFDR Article 8/9 boilerplate?
These phrases are required regulatory language and are NEVER an objective on their own:
- "promotes environmental and/or social characteristics"
- "is promoting ESG characteristics"
- "is classified as Article 8 under SFDR"
If the text contains ONLY this boilerplate with no additional specifics, there is no sustainable objective.

Step 2 — Does the text go beyond boilerplate with a specific sustainable commitment?
If boilerplate language is followed by or combined with a specific commitment, extract the commitment as a sustainable objective. The boilerplate framing is excluded; the specific commitment is extracted.

EXTRACT as sustainable objectives:
- "maintains a minimum share of sustainable investments" → sustainable objective
- "solidarity investments of 5-10% in approved solidarity enterprises" → sustainable objective
- "integrating criteria for good governance and sustainable development" → sustainable objective
- "lower carbon intensity than the benchmark index" → sustainable objective
- "reduced greenhouse gas emissions through specific targets" → sustainable objective
- "higher ESG score than the index" → sustainable objective
- "contribute to reducing greenhouse gas emissions" → sustainable objective
- "positive impact on environment and social objectives" → sustainable objective
- Specific sector exclusions framed as a goal (e.g. "exclusion of tobacco, weapons, fossil fuels") → sustainable objective

Do NOT extract as sustainable objectives:
- "taking into account ESG criteria" → approach, not outcome
- "considering sustainability risks" → process, not objective
- "ESG integration in the investment process" → methodology, not objective
- "the fund employs ESG criteria in stock selection" → screening method, not objective
- Generic "promotes environmental and/or social characteristics" without any specifics attached

Step 3 — Does the fund explicitly disclaim sustainable objectives?
If the text states "the fund does not have sustainable investment as its objective" and the sustainability content is framed purely as an approach or consideration (not as a commitment or target), do not extract a sustainable objective.

The key test: Does the text describe something the fund COMMITS TO ACHIEVING (an outcome, a target, a minimum allocation) or something the fund TAKES INTO ACCOUNT (a process, a consideration, a methodology)? Extract the former, exclude the latter.

Worked examples:
- "The fund promotes environmental and social characteristics and maintains a minimum share of sustainable investments under Article 8." → Exclude the boilerplate. EXTRACT "maintains a minimum share of sustainable investments" as sustainable objective.
- "The fund is classified as Article 8 under SFDR and promotes environmental and/or social characteristics." → Pure boilerplate. No sustainable objective.
- "The objective is capital growth, while taking into account ESG criteria." → Only "capital growth" is an objective. "Taking into account ESG criteria" is excluded.
- "The objective is to achieve outperformance while integrating criteria for good governance and sustainable development." → TWO objectives: (1) financial: "achieve outperformance", (2) sustainable: "integrating criteria for good governance and sustainable development"
- "The fund invests 5-10% of its assets in approved solidarity enterprises." → EXTRACT as sustainable objective — fund-level allocation commitment.
- "The fund invests in companies whose products contribute to the SDGs." → Do NOT extract — company description, not fund objective.

TIME HORIZON: If stated, include it (e.g. "over a rolling five-year period").

EXTRACTION RULES:
- Extract text VERBATIM from the source — do not paraphrase
- Classify each objective as "financial" or "sustainable"
- If no objective can be identified in a column, return an empty list for that column
- Detect the language of each column and record it
- If the column is non-English, ALSO provide an English translation of each extracted objective

OUTPUT FORMAT:
Return a JSON object where each key is the exact column name, and the value is:
{
  "language": "English" or "French" or "German" etc.,
  "objectives": [
    {
      "objective_text": "exact verbatim text from source",
      "objective_text_english": "English translation (same as objective_text if already English)",
      "objective_type": "financial" or "sustainable"
    }
  ]
}

If a column has no identifiable objective:
{
  "language": "English",
  "objectives": []
}

IMPORTANT: When extracting verbatim text that contains quotation marks (including German „..." quotes, French «...» quotes, or any other quotation marks), replace them with single quotes in the objective_text field. This is critical to ensure valid JSON output.

"""

In [4]:
# === Section 3: Few-Shot Examples (identical to your 100-fund notebook) ===

PASS1_FEW_SHOT = [
    {
        "fund_name": "Example Multi-Column Fund",
        "columns": {
            "PRIIPS KID Objective": "The Fund aims to maximise the return on your investment through a combination of capital growth and income on the Fund's assets and invest in a manner consistent with the principles of environmental, social and governance (ESG) investing. The Fund invests globally at least 70% of its total assets in the equity securities of companies the main business of which is financial services.",
            "PRIIPS KID Objective - French": "Le Fonds vise à maximiser le rendement de votre investissement par une combinaison de croissance du capital et de revenus sur les actifs du Fonds et à investir d'une manière conforme aux principes de l'investissement environnemental, social et de gouvernance (ESG). Le Fonds investit à l'échelle mondiale au moins 70 % de son actif total dans les titres de participation de sociétés dont l'activité principale est les services financiers."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [{"objective_text": "maximise the return on your investment through a combination of capital growth and income", "objective_text_english": "maximise the return on your investment through a combination of capital growth and income", "objective_type": "financial"}]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [{"objective_text": "maximiser le rendement de votre investissement par une combinaison de croissance du capital et de revenus", "objective_text_english": "maximise the return on your investment through a combination of capital growth and income", "objective_type": "financial"}]
            }
        }
    },
    {
        "fund_name": "Example Sustainability Split Fund",
        "columns": {
            "PRIIPS KID Objective": "The fund seeks to achieve capital growth and to outperform the benchmark. The fund's sustainable investment objective is to contribute to reducing greenhouse gas emissions. The fund also aims to have long-term positive impact on environment and social objectives."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {"objective_text": "achieve capital growth", "objective_text_english": "achieve capital growth", "objective_type": "financial"},
                    {"objective_text": "outperform the benchmark", "objective_text_english": "outperform the benchmark", "objective_type": "financial"},
                    {"objective_text": "contribute to reducing greenhouse gas emissions", "objective_text_english": "contribute to reducing greenhouse gas emissions", "objective_type": "sustainable"},
                    {"objective_text": "have long-term positive impact on environment and social objectives", "objective_text_english": "have long-term positive impact on environment and social objectives", "objective_type": "sustainable"}
                ]
            }
        }
    },
    {
        "fund_name": "Example Norwegian Fund",
        "columns": {
            "PRIIPS KID Objective": "Målsetting\n\nFondets målsetting er å skape høyest mulig relativ avkastning mot referanseindeksen, MSCI World AC, Net Total Return (målt i NOK).\n\nFondet skal investere i selskaper globalt som har løsninger på FN's bærekraftsmål og dermed bidrar til omstillingen til et mer bærekraftig samfunn."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "Norwegian",
                "objectives": [{"objective_text": "skape høyest mulig relativ avkastning mot referanseindeksen, MSCI World AC, Net Total Return (målt i NOK)", "objective_text_english": "create the highest possible relative return against the benchmark index, MSCI World AC, Net Total Return (measured in NOK)", "objective_type": "financial"}]
            }
        }
    },
    {
        "fund_name": "Example No-Objective Fund",
        "columns": {
            "PRIIPS KID Objective": "Management objective: Management takes as reference the profitability of the EUROSTOXX 50 Index, solely for informational or comparative purposes. Investment policy: Will invest more than 75% of total exposure in equity assets of European issuers."
        },
        "response": {
            "PRIIPS KID Objective": {"language": "English", "objectives": []}
        }
    },
    {
        "fund_name": "Example Company-Activity Exclusion Fund",
        "columns": {
            "KIID Objective/Investment Policy": "The fund aims to provide capital growth over the long term (5 years or more) by investing in US companies whose products and services are considered by the investment manager as contributing to positive environmental or social change and thereby have an impact on the development of a sustainable global economy."
        },
        "response": {
            "KIID Objective/Investment Policy": {
                "language": "English",
                "objectives": [{"objective_text": "provide capital growth over the long term (5 years or more)", "objective_text_english": "provide capital growth over the long term (5 years or more)", "objective_type": "financial"}]
            }
        }
    }
]

In [5]:
# === Section 4: Helpers ===

def get_nonempty_columns(row, objective_columns):
    """Return dict of only columns that have real content."""
    columns = {}
    for col in objective_columns:
        if col in row.index:
            value = row[col]
            if pd.notna(value) and str(value).strip() not in ['-', 'Not available', '']:
                columns[col] = str(value)
    return columns


def build_messages(fund_name, fund_id, columns_dict):
    """Build the full message list (few-shot + real request) for one fund."""
    columns_text = "\n\n".join(
        [f"=== Column: {col} ===\n{val}" for col, val in columns_dict.items()]
    )
    user_prompt = f"Fund ID: {fund_id}\nFund Name: {fund_name}\n\n{columns_text}"

    messages = []
    for ex in PASS1_FEW_SHOT:
        ex_text = "\n\n".join(
            [f"=== Column: {col} ===\n{val}" for col, val in ex["columns"].items()]
        )
        messages.append({"role": "user", "content": f"Fund Name: {ex['fund_name']}\n\n{ex_text}"})
        messages.append({"role": "assistant", "content": json.dumps(ex["response"], indent=2)})
    messages.append({"role": "user", "content": user_prompt})
    return messages


def robust_json_parse(text):
    """Multi-stage JSON parser for LLM output with European multilingual text."""
    if not isinstance(text, str):
        return None
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r'^```\w*\n?', '', cleaned)
        cleaned = re.sub(r'\n?```\s*$', '', cleaned)
        cleaned = cleaned.strip()
    for char, esc in [('\u201E','\\u201E'),('\u201C','\\u201C'),('\u201D','\\u201D'),
                      ('\u00AB','\\u00AB'),('\u00BB','\\u00BB'),('\u201A','\\u201A'),
                      ('\u2018','\\u2018'),('\u2019','\\u2019')]:
        cleaned = cleaned.replace(char, esc)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    def fix_strings(match):
        s = match.group(0)
        s = s.replace('\n', '\\n').replace('\r', '\\r').replace('\t', '\\t')
        return s
    fixed = re.sub(r'"(?:[^"\\]|\\.)*"', fix_strings, cleaned, flags=re.DOTALL)
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass
    brace_match = re.search(r'\{.*\}', fixed, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group(0))
        except json.JSONDecodeError:
            pass
    return None

In [6]:
# === Section 5: Build & Submit Batch ===

print("Loading data...")
df = pd.read_excel(INPUT_FILE)
print(f"  Total funds: {len(df)}")

# Build batch requests
batch_requests = []
fund_metadata = {}  # custom_id → {FundId, Fund_Name, columns_sent}

for idx in range(len(df)):
    row = df.iloc[idx]
    fund_id = str(row['FundId'])
    fund_name = row['Name']
    columns_dict = get_nonempty_columns(row, OBJECTIVE_COLUMNS)

    if not columns_dict:
        continue

    # custom_id must be alphanumeric, hyphens, underscores, max 64 chars
    custom_id = f"fund-{fund_id}"
    # Sanitise: replace anything invalid with underscore
    custom_id = re.sub(r'[^a-zA-Z0-9_-]', '_', custom_id)[:64]

    messages = build_messages(fund_name, fund_id, columns_dict)

    batch_requests.append(
        Request(
            custom_id=custom_id,
            params=MessageCreateParamsNonStreaming(
                model=MODEL,
                max_tokens=8000,
                temperature=0,
                system=PASS1_SYSTEM_PROMPT,
                messages=messages
            )
        )
    )

    fund_metadata[custom_id] = {
        'FundId': row['FundId'],
        'Fund_Name': fund_name,
        'columns_sent': list(columns_dict.keys()),
        'num_columns_sent': len(columns_dict)
    }

print(f"  Requests built: {len(batch_requests)}")
print(f"  Funds with no columns (skipped): {len(df) - len(batch_requests)}")

# Save metadata mapping so we can match results later (survives kernel restart)
metadata_path = os.path.join(OUTPUT_DIR, 'batch_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(fund_metadata, f)
print(f"  Metadata saved to {metadata_path}")

Loading data...
  Total funds: 5680
  Requests built: 5667
  Funds with no columns (skipped): 13
  Metadata saved to /Users/dannyhogan/Desktop/Hogan_RA_Work/batch_metadata.json


In [7]:
# === Section 5b: Submit the Batch ===
# This is the API call. After this, you can close the notebook.

print(f"Submitting batch of {len(batch_requests)} requests...")
print("(This may take 1-2 minutes to upload)")

message_batch = client.messages.batches.create(
    requests=batch_requests
)

BATCH_ID = message_batch.id
print(f"\nBatch submitted successfully!")
print(f"  Batch ID: {BATCH_ID}")
print(f"  Status: {message_batch.processing_status}")
print(f"  Created: {message_batch.created_at}")
print(f"  Expires: {message_batch.expires_at}")
print(f"\n  >>> SAVE THIS BATCH ID: {BATCH_ID}")
print(f"  >>> You can now close this notebook and come back later.")

# Also save batch ID to disk
batch_id_path = os.path.join(OUTPUT_DIR, 'batch_id.txt')
with open(batch_id_path, 'w') as f:
    f.write(BATCH_ID)
print(f"  Batch ID saved to {batch_id_path}")

Submitting batch of 5667 requests...
(This may take 1-2 minutes to upload)

Batch submitted successfully!
  Batch ID: msgbatch_011kkZt8nUokSNAqTtrvv2XV
  Status: in_progress
  Created: 2026-05-28 15:50:02.784679+00:00
  Expires: 2026-05-29 15:50:02.784679+00:00

  >>> SAVE THIS BATCH ID: msgbatch_011kkZt8nUokSNAqTtrvv2XV
  >>> You can now close this notebook and come back later.
  Batch ID saved to /Users/dannyhogan/Desktop/Hogan_RA_Work/batch_id.txt


In [10]:
# === Section 6: Check Status (run anytime) ===
# If you restarted the kernel, this loads the batch ID from disk.

batch_id_path = os.path.join(OUTPUT_DIR, 'batch_id.txt')
with open(batch_id_path, 'r') as f:
    BATCH_ID = f.read().strip()

status = client.messages.batches.retrieve(BATCH_ID)

print(f"Batch: {BATCH_ID}")
print(f"Status: {status.processing_status}")
print(f"Counts:")
print(f"  Processing: {status.request_counts.processing}")
print(f"  Succeeded:  {status.request_counts.succeeded}")
print(f"  Errored:    {status.request_counts.errored}")
print(f"  Canceled:   {status.request_counts.canceled}")
print(f"  Expired:    {status.request_counts.expired}")
total = (status.request_counts.processing + status.request_counts.succeeded +
         status.request_counts.errored + status.request_counts.canceled +
         status.request_counts.expired)
done = total - status.request_counts.processing
if total > 0:
    print(f"  Progress:   {done}/{total} ({done/total*100:.1f}%)")
if status.processing_status == 'ended':
    print(f"\n  BATCH COMPLETE — run Section 7 to download results.")

Batch: msgbatch_011kkZt8nUokSNAqTtrvv2XV
Status: ended
Counts:
  Processing: 0
  Succeeded:  5654
  Errored:    13
  Canceled:   0
  Expired:    0
  Progress:   5667/5667 (100.0%)

  BATCH COMPLETE — run Section 7 to download results.


In [11]:
# === Section 7: Download & Parse Results ===
# Only run this after Section 6 shows status = 'ended'

# Load metadata
metadata_path = os.path.join(OUTPUT_DIR, 'batch_metadata.json')
with open(metadata_path, 'r') as f:
    fund_metadata = json.load(f)

batch_id_path = os.path.join(OUTPUT_DIR, 'batch_id.txt')
with open(batch_id_path, 'r') as f:
    BATCH_ID = f.read().strip()

# Stream results
print(f"Downloading results for batch {BATCH_ID}...")
all_results = []
succeeded = 0
errored = 0
expired = 0
total_input_tokens = 0
total_output_tokens = 0

for result in client.messages.batches.results(BATCH_ID):
    custom_id = result.custom_id
    meta = fund_metadata.get(custom_id, {'FundId': custom_id, 'Fund_Name': 'UNKNOWN',
                                          'columns_sent': [], 'num_columns_sent': 0})

    if result.result.type == 'succeeded':
        succeeded += 1
        msg = result.result.message
        total_input_tokens += msg.usage.input_tokens
        total_output_tokens += msg.usage.output_tokens

        # Parse the response text
        response_text = msg.content[0].text if msg.content else ''
        parsed = robust_json_parse(response_text)
        if parsed is None:
            parsed = {'_error': f'JSON parse fail: {response_text[:300]}'}

        all_results.append({
            'FundId': meta['FundId'],
            'Fund_Name': meta['Fund_Name'],
            'columns_sent': meta['columns_sent'],
            'num_columns_sent': meta['num_columns_sent'],
            'pass1_raw': parsed,
            'input_tokens': msg.usage.input_tokens,
            'output_tokens': msg.usage.output_tokens
        })

    elif result.result.type == 'errored':
        errored += 1
        error_msg = str(result.result.error) if result.result.error else 'Unknown error'
        all_results.append({
            'FundId': meta['FundId'],
            'Fund_Name': meta['Fund_Name'],
            'columns_sent': meta['columns_sent'],
            'num_columns_sent': meta['num_columns_sent'],
            'pass1_raw': {'_error': error_msg},
            'input_tokens': 0,
            'output_tokens': 0
        })

    elif result.result.type == 'expired':
        expired += 1
        all_results.append({
            'FundId': meta['FundId'],
            'Fund_Name': meta['Fund_Name'],
            'columns_sent': meta['columns_sent'],
            'num_columns_sent': meta['num_columns_sent'],
            'pass1_raw': {'_error': 'Request expired (24h timeout)'},
            'input_tokens': 0,
            'output_tokens': 0
        })

print(f"\nResults downloaded:")
print(f"  Succeeded: {succeeded}")
print(f"  Errored:   {errored}")
print(f"  Expired:   {expired}")
print(f"  Total:     {len(all_results)}")
print(f"\nTokens — input: {total_input_tokens:,}, output: {total_output_tokens:,}")
batch_cost = (total_input_tokens * 1.5 / 1_000_000) + (total_output_tokens * 7.5 / 1_000_000)
print(f"Estimated cost (batch pricing): ~${batch_cost:.2f}")


Results downloaded:
  Succeeded: 5654
  Errored:   13
  Expired:   0
  Total:     5667

Tokens — input: 46,931,934, output: 7,151,809
Estimated cost (batch pricing): ~$124.04


In [12]:
# === Section 8: Summary Stats ===

pass1_df = pd.DataFrame(all_results)

error_count = 0
funds_with_obj = 0
total_obj = 0

for _, row in pass1_df.iterrows():
    raw = row['pass1_raw']
    if '_error' in raw:
        error_count += 1
        continue
    fund_obj_count = 0
    for col_name, col_data in raw.items():
        if isinstance(col_data, dict) and 'objectives' in col_data:
            fund_obj_count += len(col_data['objectives'])
    total_obj += fund_obj_count
    if fund_obj_count > 0:
        funds_with_obj += 1

print("PASS 1 SUMMARY:")
print(f"  Total funds: {len(pass1_df)}")
print(f"  Funds with errors: {error_count}")
print(f"  Funds with ≥1 objective: {funds_with_obj}")
print(f"  Total objectives extracted: {total_obj}")
print(f"  Avg objectives per fund: {total_obj / max(1, len(pass1_df) - error_count):.1f}")

PASS 1 SUMMARY:
  Total funds: 5667
  Funds with errors: 25
  Funds with ≥1 objective: 5540
  Total objectives extracted: 59745
  Avg objectives per fund: 10.6


In [13]:
# === Section 9: Save Final Output ===

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")

output_df = pass1_df.copy()
output_df['pass1_raw'] = output_df['pass1_raw'].apply(
    lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, dict) else x
)
output_df['columns_sent'] = output_df['columns_sent'].apply(
    lambda x: json.dumps(x) if isinstance(x, list) else x
)

p1_filename = f'Pass1_Extract_{len(pass1_df)}_funds_{timestamp}.xlsx'
p1_path = os.path.join(OUTPUT_DIR, p1_filename)
output_df.to_excel(p1_path, index=False, engine='openpyxl')
print(f"Saved: {p1_filename}")
print(f"  → Use this file as input to Pass 2")

Saved: Pass1_Extract_5667_funds_20260528_1706.xlsx
  → Use this file as input to Pass 2


In [ ]:
# === Section 10 (Optional): Retry Failed/Expired Requests ===
# Collects any errored or expired funds and submits them as a new batch.

failed = [r for r in all_results if isinstance(r['pass1_raw'], dict) and '_error' in r['pass1_raw']]
print(f"Found {len(failed)} failed/expired funds")

if failed:
    failed_ids = {r['FundId'] for r in failed}
    retry_requests = []

    for idx in range(len(df)):
        row = df.iloc[idx]
        if row['FundId'] not in failed_ids:
            continue
        fund_id = str(row['FundId'])
        fund_name = row['Name']
        columns_dict = get_nonempty_columns(row, OBJECTIVE_COLUMNS)
        if not columns_dict:
            continue

        custom_id = re.sub(r'[^a-zA-Z0-9_-]', '_', f"retry-{fund_id}")[:64]
        messages = build_messages(fund_name, fund_id, columns_dict)

        retry_requests.append(
            Request(
                custom_id=custom_id,
                params=MessageCreateParamsNonStreaming(
                    model=MODEL,
                    max_tokens=8000,
                    temperature=0,
                    system=PASS1_SYSTEM_PROMPT,
                    messages=messages
                )
            )
        )

    if retry_requests:
        print(f"Submitting retry batch of {len(retry_requests)} requests...")
        retry_batch = client.messages.batches.create(requests=retry_requests)
        print(f"Retry batch ID: {retry_batch.id}")
        print(f"Save this ID and check status with Section 6 (update batch_id.txt)")
    else:
        print("No retryable requests (all failures had empty columns)")